# DINOv2 — last-block fine-tuning

The actual DINOv2 ViT-S/14 architecture is in the local `dinov2_model.py` file. Only Transformer block 12 and the Low/High head are updated.


In [ ]:
import sys
import torch
from pathlib import Path

MODEL_FOLDER = (
    Path.home() / "Desktop/Paper replication/model_reproductions_7_models"
    / "05_dinov2_fine_tuned"
)
sys.path.insert(0, str(MODEL_FOLDER))

from dinov2_model import build_dinov2_small
from vit_training_helpers import configure_parameters, set_seed, train_fine_tuned_stable


In [ ]:
set_seed(42)
model = build_dinov2_small()
initial_checkpoint = (
    MODEL_FOLDER.parent / '04_dinov2_frozen/notebook_results/best_validation_accuracy.pth'
)
model.load_state_dict(torch.load(initial_checkpoint, map_location='cpu', weights_only=True))
configure_parameters(model, phase='fine_tuned', last_stage=model.blocks[-1])
print(model.blocks[-1])
print('Trainable:', sum(p.numel() for p in model.parameters() if p.requires_grad))


In [ ]:
train_fine_tuned_stable(
    model=model, last_stage=model.blocks[-1],
    output_dir=MODEL_FOLDER/'notebook_results_cpu_notebook',
    epochs=10, learning_rate=1e-5, batch_size=8, center_crop=True,
)
